# Tarang receipts · PaddleOCR-VL on Kaggle (2 × T4)

Reads every receipt photo with **PaddleOCR-VL-0.9B** and saves the raw text. Fields, scoring and decisions happen on the laptop
(`scripts/vlm_extract.py`) with the same parsing and held-out receipts as the Tesseract baselines. This notebook only **reads**.

**Why earlier runs took ~180 s per receipt:** the model's published `generation_config.json` sets `"use_cache": false`, so
generation re-processed the whole image (~2,900 patches) for *every* output token. This notebook forces the KV cache on, uses the
**native** `transformers` implementation (no `trust_remote_code`, no Paddle install, which broke PyTorch's NCCL), batches photos, and
runs **one worker per GPU**.

**Setup:** attach the `tarang-receipts` dataset · *Accelerator → GPU T4 x2* · *Internet → On* · fresh session
(*Run → Factory reset* if this session ever installed Paddle) · *Run All*.

**Output:** `/kaggle/working/paddleocr_vl_outputs.jsonl` → download → `data/interim/vlm/`.

In [1]:
# Settings
BATCH_SIZE = 8            # photos per GPU call; lower to 4 or 2 on CUDA out-of-memory
DTYPE = "auto"            # "auto": bfloat16 on GPUs with native bf16 (compute >= 8.0), else float32.
                          # A T4 *reports* bf16 support but emulates it. "float16" is faster on a T4; use it only if the
                          # speed-check text looks right (fp16 can overflow).
MAX_NEW_TOKENS = 768      # receipts need ~150-300 tokens; the cap only stops a runaway loop
MAX_IMAGES = None         # e.g. 16 for a smoke test; None = all photos
TRANSFORMERS_VERSION = "5.19.0"   # has native PaddleOCRVLForConditionalGeneration

In [2]:
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv

index, name, memory.total [MiB], compute_cap
0, Tesla T4, 15360 MiB, 7.5
1, Tesla T4, 15360 MiB, 7.5


## Install (transformers only; no Paddle)

In [3]:
import subprocess, sys

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"transformers=={TRANSFORMERS_VERSION}", "accelerate"],
                   capture_output=True, text=True)
print("pip:", "ok" if r.returncode == 0 else r.stderr[-1500:])
check = subprocess.run([sys.executable, "-c", "import torch, transformers; from transformers import PaddleOCRVLForConditionalGeneration; "
                        "print(torch.__version__, torch.cuda.device_count(), 'GPUs', transformers.__version__)"],
                       capture_output=True, text=True)
if check.returncode:
    print(check.stderr[-1500:])
    raise RuntimeError("torch/transformers broken in this session. Run -> Factory reset, then Run All (do not install Paddle).")
print("torch, GPUs, transformers:", check.stdout.strip())

pip: ok
torch, GPUs, transformers: 2.11.0+cu128 2 GPUs 5.19.0


In [4]:
import glob, json, os, time
from pathlib import Path

import pandas as pd

jpgs = sorted(glob.glob("/kaggle/input/**/*.jpg", recursive=True))
assert jpgs, "No .jpg files under /kaggle/input. Attach the tarang-receipts dataset (Add Input -> Datasets)."
audit_csv = next(iter(glob.glob("/kaggle/input/**/image_audit.csv", recursive=True)), "")
WORK = Path("/kaggle/working")
OUT_JSONL = WORK / "paddleocr_vl_outputs.jsonl"
print(f"{len(jpgs)} photos | audit: {audit_csv or 'missing (no rotation fix)'}")

467 photos | audit: /kaggle/input/datasets/rishi412/tarang-receipts/image_audit.csv


## The worker

One process per GPU. Each loads the model, rotates sideways photos upright (rotation measured in our image audit: OCR at
0/90/180/270°, all 30 verified by eye; recorded as `rotation_applied`), reads its share of photos in batches, and appends to its
own shard file. Re-running skips photos already done.

In [5]:
WORKER = r"""
import argparse, json, sys, time
from pathlib import Path
import pandas as pd, torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

ap = argparse.ArgumentParser()
ap.add_argument("--files"); ap.add_argument("--out"); ap.add_argument("--audit", default="")
ap.add_argument("--batch", type=int); ap.add_argument("--dtype"); ap.add_argument("--max-new", type=int)
ap.add_argument("--probe", action="store_true")
a = ap.parse_args()

files = json.loads(Path(a.files).read_text())
rotation = {}
if a.audit:
    au = pd.read_csv(a.audit, dtype=str, keep_default_na=False)
    rotation = {r.image_file: int(float(r.rotation)) for r in au.itertuples() if r.rotation not in ("", "nan")}

cap = torch.cuda.get_device_capability(0)
dtype = (torch.bfloat16 if cap >= (8, 0) else torch.float32) if a.dtype == "auto" else getattr(torch, a.dtype)
MODEL_ID = "PaddlePaddle/PaddleOCR-VL"
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=dtype).to("cuda").eval()
model.generation_config.use_cache = True     # the published config says False: that made generation ~50x slower
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"
backend = f"paddleocr-vl-0.9b-hf-native-{str(dtype).split('.')[-1]}"
pad = processor.tokenizer.pad_token_id
print(f"[{torch.cuda.get_device_name(0)}] {type(model).__name__} {dtype} {backend}", flush=True)

def read(names):
    ims, angles = [], []
    for n in names:
        ang = rotation.get(Path(n).name, 0)
        ims.append(Image.open(n).convert("RGB").rotate(ang, expand=True)); angles.append(ang)
    batch = [[{"role": "user", "content": [{"type": "image", "image": im}, {"type": "text", "text": "OCR:"}]}] for im in ims]
    inputs = processor.apply_chat_template(batch, add_generation_prompt=True, tokenize=True, return_dict=True,
                                           return_tensors="pt", processor_kwargs={"padding": True}).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=a.max_new, do_sample=False, use_cache=True)
    new = out[:, inputs["input_ids"].shape[1]:]
    texts = processor.batch_decode(new, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return texts, angles, [int((t != pad).sum()) for t in new]

if a.probe:
    read(files[:1])                                    # warm-up
    t0 = time.perf_counter(); texts, _, k = read(files[:2]); dt = time.perf_counter() - t0
    print(f"PROBE {dt / 2:.1f}s/photo, {sum(k) / dt:.1f} tokens/s", flush=True)
    print(texts[0][:400], flush=True)
    sys.exit(0)

out_path = Path(a.out)
done = set()
if out_path.exists():
    for line in out_path.read_text().splitlines():
        r = json.loads(line)
        if not r["error"] and r["backend"] == backend:
            done.add(r["file"])
todo = [f for f in files if Path(f).name not in done]
print(f"{len(done)} done, {len(todo)} to go", flush=True)

read(todo[:1]) if todo else None                       # warm-up, not timed
for i in range(0, len(todo), a.batch):
    names = todo[i:i + a.batch]
    t0 = time.perf_counter()
    try:
        texts, angles, k = read(names); errs = [""] * len(names)
    except Exception as e:
        torch.cuda.empty_cache()
        texts, angles, k = [""] * len(names), [None] * len(names), [0] * len(names)
        errs = [f"{type(e).__name__}: {e}"[:500]] * len(names)
    dt = time.perf_counter() - t0
    with out_path.open("a") as f:
        for n, t, ang, kk, err in zip(names, texts, angles, k, errs):
            f.write(json.dumps({"file": Path(n).name, "backend": backend, "text": t, "rotation_applied": ang,
                                "new_tokens": kk, "batch_size": len(names), "batch_seconds": round(dt, 3),
                                "seconds": round(dt / len(names), 3), "error": err}, ensure_ascii=False) + "\n")
    print(f"PROGRESS {min(i + a.batch, len(todo))}/{len(todo)} {dt / len(names):.2f}s/photo", flush=True)
print("DONE", flush=True)
"""
(WORK / "worker.py").write_text(WORKER)

def worker_cmd(gpu, files, out, probe=False):
    lst = WORK / f"files_gpu{gpu}.json"
    lst.write_text(json.dumps(files))
    cmd = [sys.executable, str(WORK / "worker.py"), "--files", str(lst), "--out", str(out), "--audit", audit_csv,
           "--batch", str(BATCH_SIZE), "--dtype", DTYPE, "--max-new", str(MAX_NEW_TOKENS)] + (["--probe"] if probe else [])
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return cmd, env

## Speed check on GPU 0 (2 photos)

Should be a few seconds per photo. If it's still above ~60 s, **stop** and send this cell's output.

In [6]:
cmd, env = worker_cmd(0, jpgs[:2], WORK / "probe.jsonl", probe=True)
r = subprocess.run(cmd, env=env, capture_output=True, text=True)
print("\n".join(l for l in r.stdout.splitlines()))
if r.returncode:
    print(r.stderr[-3000:])
    raise RuntimeError("worker failed; send this output")
import re
seconds_per_photo = float(re.search(r"PROBE ([0-9.]+)s/photo", r.stdout).group(1))
if seconds_per_photo > 60:
    raise RuntimeError("Still too slow; send this output. Try DTYPE='float16' or BATCH_SIZE=4.")

[Tesla T4] PaddleOCRVLForConditionalGeneration torch.float32 paddleocr-vl-0.9b-hf-native-float32
PROBE 3.5s/photo, 54.6 tokens/s
BPCL
LAXMI AUTO FUELS
Arvi Naka, Wardha
GSTIN: 27DPAFU4120X1ZM

Receipt No: 003350
Date: 11/06/26 Time: 17:19
Pump: 4 Nozzle: 4
Product: PETROL
Rate (Rs/L): 104.69
Volume (L): 2.87
Amount (Rs): 300.00
Vehicle No: MH32AY2051
Mode: Cash

TOTAL Rs. 300.00

Thank You! Visit Again
Save Fuel, Save Money


## Run all photos on both GPUs

In [7]:
import torch
n_gpu = max(1, torch.cuda.device_count())
todo = jpgs[:MAX_IMAGES]
shards = [todo[g::n_gpu] for g in range(n_gpu)]          # interleaved so both GPUs get a similar mix
procs = []
for g in range(n_gpu):
    cmd, env = worker_cmd(g, shards[g], WORK / f"paddleocr_vl_gpu{g}.jsonl")
    log = open(WORK / f"worker_gpu{g}.log", "w")
    procs.append((subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log))
print(f"{n_gpu} workers started: {[len(s) for s in shards]} photos each")

from tqdm.auto import tqdm
def count(g):
    p = WORK / f"paddleocr_vl_gpu{g}.jsonl"
    return sum(1 for _ in p.open()) if p.exists() else 0
with tqdm(total=len(todo), unit="photo") as bar:   # counts shard lines, so photos done in an earlier session show at once
    while any(p.poll() is None for p, _ in procs):
        time.sleep(10)
        bar.n = min(len(todo), sum(count(g) for g in range(n_gpu))); bar.refresh()
for g, (p, log) in enumerate(procs):
    log.close()
    print(f"GPU {g} exit code {p.returncode}; last log lines:")
    print("".join(open(WORK / f"worker_gpu{g}.log").readlines()[-3:]))

2 workers started: [234, 233] photos each


  0%|          | 0/467 [00:00<?, ?photo/s]

GPU 0 exit code 0; last log lines:
PROGRESS 232/234 6.65s/photo
PROGRESS 234/234 13.61s/photo
DONE

GPU 1 exit code 0; last log lines:
PROGRESS 232/233 2.73s/photo
PROGRESS 233/233 5.99s/photo
DONE



## Merge and check

In [8]:
rows = []
for g in range(n_gpu):
    p = WORK / f"paddleocr_vl_gpu{g}.jsonl"
    if p.exists():
        rows += [json.loads(l) for l in p.read_text().splitlines()]
out = pd.DataFrame(rows).drop_duplicates("file", keep="last").sort_values("file")
out.to_json(OUT_JSONL, orient="records", lines=True, force_ascii=False)

ok = out[out.error == ""]
print(f"{len(out)} photos, {len(ok)} ok, {(out.error != '').sum()} errors -> {OUT_JSONL}")
print(f"seconds/photo per GPU (batch time / batch size): median {ok.seconds.median():.2f}")
print(f"new tokens/photo: median {ok.new_tokens.median():.0f}, max {ok.new_tokens.max()}; hit the cap: {(ok.new_tokens >= MAX_NEW_TOKENS).sum()}")
print(out[out.error != ""][["file", "error"]].head())
for _, r in ok.head(2).iterrows():
    print("\n=====", r.file, "\n", r.text[:600])

467 photos, 467 ok, 0 errors -> /kaggle/working/paddleocr_vl_outputs.jsonl
seconds/photo per GPU (batch time / batch size): median 2.87
new tokens/photo: median 151, max 768; hit the cap: 11
Empty DataFrame
Columns: [file, error]
Index: []

===== IMG-20260610-WA0001.jpg 
 BPCL
LAXMI AUTO FUELS
Arvi Naka, Wardha
GSTIN: 27DPAFU4120X1ZM

Receipt No: 003350
Date: 11/06/26 Time: 17:19
Pump: 4 Nozzle: 4
Product: PETROL
Rate (Rs/L): 104.69
Volume (L): 2.87
Amount (Rs): 300.00
Vehicle No: MH32AY2051
Mode: Cash

TOTAL Rs. 300.00

Thank You! Visit Again
Save Fuel, Save Money

===== IMG-20260612-WA0058.jpg 
 Spice Route Kitchen
Civil Lines, Nagpur
Ph: 0712-2765565
GSTIN: 27NCRPN1784A3Z1
TAX INVOICE

Bill No: 5726
Date: 09-06-2026
Steward: Kiran

Item
Qty
Amount

Veg Biryani
1
240.00
Tandoori Roti
2
50.00
Butter Roti
2
60.00
Sub Total
350.00
CGST @2.5%
8.75
SGST @2.5%
8.75
Round Off
+0.50

GRAND TOTAL Rs
368.00
for claim
Items: 3
Pay Mode: UPI
Thank you, visit again!
